# Notebook 3 -- Multidimensional Deep Queue-Reactive (MDQR) Model

**Paper:** *Deep Learning Meets Queue-Reactive: A Framework for Realistic Limit Order Book Simulation* (Bodor & Carlier, 2025)

**Objective.** Notebooks 1 and 2 model each queue *independently*. This notebook implements
the MDQR model (Section 4), which treats the entire limit order book as a single multidimensional
entity. The key contributions over DQR are:

1. **Joint intensity model** — one network outputs $3 \times 2K = 30$ intensities simultaneously,
   enabling the model to capture cross-level and cross-side dependencies.
2. **Order-size model (SizeNet)** — a separate network models the distribution of order sizes
   conditional on event type, level, and LOB state.
3. **Simulation** with calibrated intensities and sizes, producing synthetic LOB trajectories.

Validation figures (Figures 5, 9--18 of the paper) confirm that the simulator reproduces key
stylized facts of limit order book dynamics.

### Key differences from DQR (Notebook 2)

| Property | DQR (NB2) | MDQR (NB3) |
|----------|-----------|-----------|
| Number of networks | 10 (one per queue) | 1 (shared) |
| Output size | 3 (L/C/M per queue) | 30 ($3 \times 2K$, all queues) |
| Cross-level information in $\mathbf{x}_k$ | No | Yes (all $2K$ queue sizes) |
| Inter-event time $\Delta t_k$ | Per-level (Section 3.2) | **Global** (Section 4.2) |
| Order-size model | No | Yes (SizeNet, cross-entropy) |
| Architecture | [128, 32] + BN | [256, 64], no BN |

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy import stats
from scipy.stats import gamma as gamma_dist

from lobster import (load_lobster_data, compute_aes_by_level,
                     normalize_by_aes, make_descriptive_table)

# ── HARDWARE DETECTION (CUDA / MPS / CPU) ─────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using device: CUDA  ({torch.cuda.get_device_name(0)})")
    props = torch.cuda.get_device_properties(0)
    print(f"  VRAM: {props.total_memory/1024**3:.1f} GB | CUDA {props.major}.{props.minor}")
    torch.backends.cudnn.benchmark = True
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using device: Apple Silicon (MPS)")
    print("  Hardware acceleration enabled for Mac architectures.")
else:
    device = torch.device('cpu')
    print("Using device: CPU")
    print("  NOTE: Training will be slow on CPU; reduce EPOCHS if needed.")

# ── PLOTTING & REPRODUCIBILITY CONFIGURATION ──────────────────────────────────
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 12
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3

np.random.seed(42)
torch.manual_seed(42)

Using device: CUDA  (NVIDIA GeForce GTX 1650 with Max-Q Design)
  VRAM: 4.0 GB | CUDA 7.5


---
## 1. Data Loading and Preprocessing

We use the same INTC LOBSTER dataset and preprocessing pipeline as Notebook 2:

1. **Reference price** $p_{\text{ref}}(t)$: disambiguates half-tick price levels when the spread is even (Section 2.1 of the paper).
2. **QR-grid regridding**: maps raw LOBSTER price levels to the $\pm 1, \ldots, \pm K$ grid centred on $p_{\text{ref}}$.
3. **AES normalisation**: $\tilde{q}_i \leftarrow \lceil q_i / \text{AES}_i \rceil$ transforms queue sizes into small integers (≈1–10), making gradient magnitudes stable across levels.

The shared-network design of MDQR requires that all $2K$ queue sizes are on a **consistent scale** — impossible without AES normalisation, since level-1 queues are typically an order of magnitude larger than level-5 queues.

In [28]:
PRIMARY  = "INTC"
DATE     = "2012-06-21"
LEVELS   = 5
K        = 5
MARKET_OPEN_S  = 34200   # 09:30 in seconds since midnight
MARKET_CLOSE_S = 57600   # 16:00

EVENT_TO_IDX = {'L': 0, 'C': 1, 'M': 2}
IDX_TO_EVENT = {0: 'L', 1: 'C', 2: 'M'}

DATA_DIR     = Path("data")
stock_folder = DATA_DIR / f"LOBSTER_SampleFile_{PRIMARY}_{DATE}_5"
msg_path = stock_folder / f"{PRIMARY}_{DATE}_34200000_57600000_message_{LEVELS}.csv"
ob_path  = stock_folder / f"{PRIMARY}_{DATE}_34200000_57600000_orderbook_{LEVELS}.csv"

msg, ob, qr, df = load_lobster_data(
    str(msg_path), str(ob_path),
    levels=LEVELS, K=K, market_open_seconds=MARKET_OPEN_S
)
aes       = compute_aes_by_level(df, K=K)
tick_size = int(qr['tick_size'].iloc[0])
df_norm   = normalize_by_aes(df, aes=aes, K=K)

print("Descriptive statistics (cf. Table 1 of the paper):")
print(make_descriptive_table(df, K=K))
print(f"\nTotal events (all levels): {len(df_norm):,}")
print(f"AES by level: {aes.values}")
print(f"Tick size: {tick_size}")

Descriptive statistics (cf. Table 1 of the paper):
       #L (×10^3)  #C (×10^3)  #M (×10^2)     AES  AIT (ms)
Level                                                      
1          197.19      162.42      324.82  483.88      32.5
2           40.20       59.75        0.00  441.55      45.6
3           22.71       22.55        0.00  451.23      71.5
4           13.23       11.88        0.00  475.07      67.4
5            9.50        8.44        0.00  596.93      65.6

Total events (all levels): 580,351
AES by level: [483.87918234 441.55301651 451.2302287  475.07007485 596.92829663]
Tick size: 100


### 1.1 Global inter-event time and the LOBSTER batch-arrival artefact

A fundamental difference from DQR (Notebook 2): MDQR uses the **global** inter-event time

$$\Delta t_k = t_k - t_{k-1}$$

where $t_{k-1}$ is the timestamp of the *previous event anywhere in the LOB* (Section 4.2 of the paper).
In DQR, $\Delta t_k$ was per-level (time since the last event *at the same queue*).  The switch to
global $\Delta t_k$ is what allows $\Lambda(\mathbf{x}_k) = \sum_j \lambda_j(\mathbf{x}_k)$ to be
interpreted as the total rate of any next LOB event.

**LOBSTER batch-arrival artefact.** LOBSTER records many events at identical microsecond timestamps
($\Delta t_k \approx 0$). These correspond to exchange-level batch-processing artefacts rather than
truly simultaneous independent arrivals. For such events, the survival term $\Lambda \cdot \Delta t_k \approx 0$
and the NLL loss degenerates to a plain cross-entropy: the model is not penalised for predicting
an arbitrarily high total rate $\Lambda$.  Unconstrained, training drives all intensities towards
$+\infty$, causing simulations to generate billions of near-instantaneous events.

**Two distinct treatments** — one for training, one for evaluation:

**Training:** we clip $\Delta t_k$ to $[10^{-6}\,\text{s},\; 30\,\text{s}]$ (identical to NB2 per-level clipping).

- The **lower bound $1\,\mu\text{s}$** prevents a literal $\log 0$ in the NLL without inflating the survival term.
  For batch-arrival events ($\Delta t \approx 0$), the survival term $\Lambda \times 10^{-6}$ remains negligible:
  these events contribute **only to the cross-entropy (type prediction) part** of the NLL.
  $\Lambda$ calibration is driven by the 28% of events with genuine inter-arrivals $> 1\,\text{ms}$.
  Using $1\,\text{ms}$ instead would inflate the survival term by a factor of $\approx\!14\times$ for the
  median batch-arrival event ($\Delta t_{\text{med}} = 0.07\,\text{ms}$), biasing $\Lambda$ downward.
- The **upper bound $30\,\text{s}$** avoids instability from large gaps when the QR grid shifts
  after a reference-price change.

**Timing evaluation metric (Section 7):** the metric is computed **only on events with original
$\Delta t_k > 1\,\text{ms}$**.  These are the genuine independent inter-arrivals that approximate the
Poisson assumption; batch-arrival events ($\Delta t \approx 0$) would inflate the median timing
error to thousands of percent and are not informative for Poisson calibration.

In [29]:
DT_MIN = 1e-6   # 1 µs — prevents log(0); survival term stays negligible for batch arrivals
DT_MAX = 30.0   # 30 s — clip ceiling (reference-price change artefacts)
DT_EVAL = 1e-3  # 1 ms — filter threshold for timing evaluation metric (genuine inter-arrivals)

# ── timing diagnostics ──────────────────────────────────────────────────────
dt_raw = df_norm['delta_time'].astype(float).values
print("Global delta_time statistics (before clipping):")
print(f"  mean:   {dt_raw.mean()*1e3:.3f} ms")
print(f"  median: {np.median(dt_raw)*1e3:.3f} ms")
print(f"  <1ms:   {(dt_raw < 1e-3).mean()*100:.1f}%  (LOBSTER batch arrivals — near-zero Δt)")
print(f"  <0.1ms: {(dt_raw < 1e-4).mean()*100:.1f}%  (sub-100µs batch arrivals)")
print(f"  >1ms:   {(dt_raw >= 1e-3).mean()*100:.1f}%  (genuine inter-arrivals — drive Λ calibration)")
print(f"  >10ms:  {(dt_raw >= 1e-2).mean()*100:.1f}%")
print()
print("Training clip: [{:.0e}, {:.0f}s]  →  batch arrivals contribute to type prediction only".format(DT_MIN, DT_MAX))
print("Eval metric filter: Δt > {:.0e}s  →  genuine inter-arrivals only".format(DT_EVAL))

# hour slots (same as NB2 — relative to market open)
df_norm['hour'] = ((df_norm['time'] - MARKET_OPEN_S) // 3600).astype(int).clip(0, 6)

# event ID and global last-event encoding
# Encoding: 0=no_prev, 1=L_ask, 2=L_bid, 3=C_ask, 4=C_bid, 5=M_ask, 6=M_bid
df_norm['event_id'] = df_norm['type'].map(EVENT_TO_IDX).astype(int)
df_norm['last_event_global'] = (
    df_norm['event_id'] * 2 + (df_norm['lvl'] < 0).astype(int) + 1
).shift(1).fillna(0).astype(int)

print(f"\nHour slots: {sorted(df_norm['hour'].unique().tolist())}")
print(f"Event counts: {dict(df_norm['event_id'].value_counts().sort_index())}")

Global delta_time statistics (before clipping):
  mean:   40.320 ms
  median: 0.069 ms
  <1ms:   71.9%  (LOBSTER batch arrivals — near-zero Δt)
  <0.1ms: 55.1%  (sub-100µs batch arrivals)
  >1ms:   28.1%  (genuine inter-arrivals — drive Λ calibration)
  >10ms:  17.5%

Training clip: [1e-06, 30s]  →  batch arrivals contribute to type prediction only
Eval metric filter: Δt > 1e-03s  →  genuine inter-arrivals only

Hour slots: [0, 1, 2, 3, 4, 5, 6]
Event counts: {0: np.int64(282826), 1: np.int64(265043), 2: np.int64(32482)}


In [30]:
# ── Trade Imbalance (TI) Calculation ──────────────────────────────────────────
# This cell computes the TI features as defined in Bodor & Carlier (2025).
# TI measures the volume imbalance between buyer-initiated and seller-initiated
# market orders over multiple rolling horizons.

print("Computing Trade Imbalances (TI) over rolling windows...")

# Ensure chronological order and create a time index for pandas rolling
df_norm = df_norm.sort_values('time').reset_index(drop=True)
time_idx = pd.to_timedelta(df_norm['time'], unit='s')
df_temp = pd.DataFrame(index=time_idx)

# Volume identification: Market orders hitting Ask (buy) or Bid (sell)
df_temp['vol_buy'] = np.where((df_norm['type'] == 'M') & (df_norm['lvl'] > 0), df_norm['size'], 0.0)
df_temp['vol_sell'] = np.where((df_norm['type'] == 'M') & (df_norm['lvl'] < 0), df_norm['size'], 0.0)

# Horizons specified in the paper: 20s, 1min, 5min, 15min
for tau, name in [(20, '20s'), (60, '1min'), (300, '5min'), (900, '15min')]:
    roll_buy = df_temp['vol_buy'].rolling(f'{tau}s').sum().values
    roll_sell = df_temp['vol_sell'].rolling(f'{tau}s').sum().values
    
    total_vol = roll_buy + roll_sell
    # TI formula: (V_buy - V_sell) / (V_buy + V_sell)
    df_norm[f'TI_{name}'] = np.where(total_vol > 0, (roll_buy - roll_sell) / total_vol, 0.0)

print(f"Successfully added TI features to df_norm. New shape: {df_norm.shape}")

Computing Trade Imbalances (TI) over rolling windows...
Successfully added TI features to df_norm. New shape: (580351, 34)


---
## 2. MDQR Mathematical Formulation (Section 4)

### 2.1 Event space and factored likelihood

Each event $e_k = (\eta_k, \ell_k, \Delta t_k, s_k, \mathbf{x}_k)$ is characterised by its
type $\eta_k \in \{L, C, M\}$, price level $\ell_k \in \{-K,\ldots,-1,1,\ldots,K\}$, global
inter-event time $\Delta t_k$, order size $s_k$, and state vector $\mathbf{x}_k$.  The joint
likelihood of the sequence factorises as (Eq. 3):

$$
\mathcal{L}(\theta \mid \mathcal{E}) =
\underbrace{\prod_{k=1}^{N} e^{-\Lambda(\mathbf{x}_k;\theta)\,\Delta t_k}\,
\lambda^{(\eta_k,\ell_k)}(\mathbf{x}_k;\theta)}_{\text{intensity component}} \times
\underbrace{\prod_{k=1}^{N} p(s_k \mid \eta_k, \ell_k, \mathbf{x}_k;\phi)}_{\text{size component}}
$$

The two components are independent and estimated separately.

### 2.2 Intensity NLL (Eq. 4)

$$
\mathcal{L}_\lambda(\theta)
= \frac{1}{N}\sum_{k=1}^{N}\Bigl[
  \underbrace{\Lambda(\mathbf{x}_k;\theta)\,\Delta t_k}_{\text{survival term}}
  -\underbrace{\log\lambda^{(\eta_k,\ell_k)}(\mathbf{x}_k;\theta)}_{\text{event likelihood}}
\Bigr]
$$

where $\Lambda = \sum_{j=1}^{30}\lambda_j$ is the total event rate.  The survival term
penalises models that predict a high overall rate: for a given observed event, the model
should concentrate intensity on the correct ($\eta_k,\ell_k$) pair while keeping $\Lambda$
commensurate with the observed inter-arrival rate $1/\Delta t_k$.

### 2.3 Order-size cross-entropy (Eq. 5)

$$
\mathcal{L}_s(\phi)
= -\frac{1}{N}\sum_{k=1}^{N}\sum_{c=1}^{C}
  y_{k,c}\,\log\hat{p}_c(s_k \mid \eta_k, \ell_k, \mathbf{x}_k;\phi)
$$

where the $C$ classes are quantile bins of the normalised size distribution.

### 2.4 Event index encoding

Each of the $3 \times 2K = 30$ output neurons is assigned an index:

$$j = \text{type\_idx} \times 2K + \text{side\_idx} \times K + (\ell - 1), \quad j \in \{0,\ldots,29\}$$

with type\_idx $\in \{0{=}L,\,1{=}C,\,2{=}M\}$, side\_idx $\in \{0{=}\text{ask},\,1{=}\text{bid}\}$.

---
## 3. Neural Network Architecture (Table 4)

Both the intensity model (MDQRNet) and the size model (SizeNet) follow the template in Table 4 of the paper.

| Component | MDQRNet (intensity) | SizeNet (size) |
|-----------|---------------------|----------------|
| Input dim | $2K + 1 + 2\times2 = 15$ | $15 + 2$ (event emb.) $= 17$ |
| Hidden dims | $[256,\; 64]$ | $[256,\; 64]$ |
| Hidden activation | $\tanh$ | $\tanh$ |
| Output dim | $3 \times 2K = 30$ | $C$ (quantile bins) |
| Output activation | ReLU (ensures $\lambda_j \geq 0$) | Softmax (probability simplex) |
| Batch Normalisation | **None** | **None** |

//

> **Why no Batch Normalisation (unlike DQR in NB2)?**
> NB2 uses BN (Linear → Tanh → BN) because it stabilises training when each queue level is
> trained independently with very few samples per level.  For MDQR the dataset is larger
> (all levels jointly), but BN introduces a systematic **train/eval discrepancy**: during
> training, each mini-batch normalises activations with its own statistics; at evaluation,
> the model uses the running mean/variance accumulated over all training batches.  With only
> one trading day of data, these running statistics diverge from the true marginal statistics,
> producing a large gap between train NLL and val NLL that is an artefact of BN rather than
> overfitting.  The paper's Table 4 lists only Tanh/ReLU/Softmax — no BN — confirming this
> is the architecturally faithful choice.

In [ ]:
# ── Event index helpers ──────────────────────────────────────────────────
def encode_event_j(t_type, lvl, K):
    # Maps (event_type_str, signed_level) -> j in [0, 3*2*K - 1]
    type_idx = EVENT_TO_IDX.get(t_type, -1)
    if type_idx < 0 or lvl == 0:
        return -1
    side_idx  = 0 if lvl > 0 else 1
    level_idx = abs(lvl) - 1
    if level_idx >= K:
        return -1
    return type_idx * (2 * K) + side_idx * K + level_idx

def decode_event_j(j, K):
    # Returns (type_idx, side_idx, level_idx) where level = level_idx + 1
    type_idx  = j // (2 * K)
    rem       = j % (2 * K)
    side_idx  = rem // K
    level_idx = rem % K
    return type_idx, side_idx, level_idx


# ── MDQRNet (intensity model) ─────────────────────────────────────────────
class MDQRNet(nn.Module):
    '''
    Joint intensity network for all 3*2*K event-level pairs.
    No BatchNorm (see design note above).
    Input: continuous features + hour embedding + last-event embedding.
    Output: 3*2*K non-negative intensities (ReLU).
    '''
    def __init__(self, K=5, n_cont=11, n_hour=7, n_last=7,
                 emb_dim=2, hidden=(256, 64)):
        super().__init__()
        self.K = K
        self.hour_emb = nn.Embedding(n_hour, emb_dim)
        self.last_emb = nn.Embedding(n_last, emb_dim)
        in_d = n_cont + 2 * emb_dim
        layers, prev = [], in_d
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            prev = h
        layers += [nn.Linear(prev, 3 * 2 * K), nn.Softplus()]
        self.net = nn.Sequential(*layers)

    def forward(self, x_cont, x_hour, x_last):
        h = torch.cat([x_cont,
                        self.hour_emb(x_hour),
                        self.last_emb(x_last)], dim=1)
        return self.net(h)   # (batch, 30)


# ── SizeNet (order-size model) ────────────────────────────────────────────
class SizeNet(nn.Module):
    '''
    Categorical size model: p(size_class | x_k, event_j).
    Event index j is encoded via a learned embedding.
    '''
    def __init__(self, K=5, n_cont=11, n_hour=7, n_last=7,
                 n_event=30, emb_dim=2, n_classes=30, hidden=(256, 64)):
        super().__init__()
        self.hour_emb  = nn.Embedding(n_hour, emb_dim)
        self.last_emb  = nn.Embedding(n_last, emb_dim)
        self.event_emb = nn.Embedding(n_event + 1, emb_dim)
        in_d = n_cont + 3 * emb_dim
        layers, prev = [], in_d
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x_cont, x_hour, x_last, x_event):
        h = torch.cat([x_cont,
                        self.hour_emb(x_hour),
                        self.last_emb(x_last),
                        self.event_emb(x_event)], dim=1)
        return torch.softmax(self.net(h), dim=-1)


# ── Sanity check ──────────────────────────────────────────────────────────
_m = MDQRNet(K=K).to(device)
_xc = torch.zeros(4, 11).to(device)
_xh = torch.zeros(4, dtype=torch.long).to(device)
_xl = torch.zeros(4, dtype=torch.long).to(device)
out = _m(_xc, _xh, _xl)
print(f"MDQRNet output shape: {out.shape}  (expect [4, {3*2*K}])")
print(f"MDQRNet trainable parameters: {sum(p.numel() for p in _m.parameters()):,}")
_s = SizeNet(K=K).to(device)
print(f"SizeNet  trainable parameters: {sum(p.numel() for p in _s.parameters()):,}")
del _m, _s, _xc, _xh, _xl, out

---
## 4. Feature Engineering and Dataset Construction

### 4.1 State vector $\mathbf{x}_k$

The MDQR state vector concatenates the LOB snapshot at all levels with temporal and categorical features:

| Feature | Type | Dim | Notes |
|---------|------|-----|-------|
| $\log(1 + \tilde{Q}_i)$, $i = 1,\ldots,K$ (ask) | Numerical | $K$ | AES-normalised ask queue sizes |
| $\log(1 + \tilde{Q}_{-i})$, $i = 1,\ldots,K$ (bid) | Numerical | $K$ | AES-normalised bid queue sizes |
| Spread proxy | Numerical | 1 | Set to 0 (INTC: $\approx$ constant 1-tick spread) |
| Hour bucket $h_k \in \{0,\ldots,6\}$ | Categorical → embed | 2 | 7 slots of 1 h from 09:30 |
| Global last-event $(η_{k-1},\,\text{side}_{k-1})$ | Categorical → embed | 2 | 7 values: 0=none, 1=L$_\text{ask}$, …, 6=M$_\text{bid}$ |

**Total:** $n_\text{cont} = 2K+1 = 11$ continuous features + 2 embeddings of dim 2 $\Rightarrow$ input dim $d = 15$.

The log-transform $\log(1+\tilde{q})$ on AES-normalised queues ensures inputs are bounded and approximately symmetric, which stabilises gradient flow through the Tanh layers.

> **Compared to the paper ($d = 25$):** the full model adds trade imbalance at 4 horizons (not available from raw LOBSTER message data) and per-level last-event embeddings (impractical with a single trading day — the outer levels would see too few events for meaningful embeddings). Our simplified $d = 15$ vector retains the most informative features.

### 4.2 Train/validation split

We use a **random 80/20 split** — identical strategy to Notebook 2.

The NLL objective $\mathcal{L}_\lambda(\theta) = \frac{1}{N}\sum_k [\Lambda(\mathbf{x}_k)\Delta t_k - \log\lambda^{(\eta_k,\ell_k)}(\mathbf{x}_k)]$
treats each event as *conditionally independent* given its state vector $\mathbf{x}_k$.  Under
this assumption, random shuffling introduces **no data leakage**: the state $\mathbf{x}_k$ already
encodes the relevant history through $h_k$ and the last-event embedding.

A chronological split would systematically exclude end-of-day patterns from training (and over-represent
them in validation), making the validation loss a biased proxy for generalisation with only one trading day.

In [ ]:
# ── Section 4: Dataset Construction ──────────────────────────────────────────

def build_mdqr_dataset(df_n, K):
    """
    Constructs the MDQR dataset by extracting continuous features,
    categorical embeddings, event labels, inter-arrival times, and sizes.
    """
    # Convert Int64 (pandas nullable) to float64 before .values to get numpy arrays
    q_ask = np.column_stack([df_n[f'Q_{i}'].fillna(0).astype(float).values  for i in range(1, K+1)])
    q_bid = np.column_stack([df_n[f'Q_-{i}'].fillna(0).astype(float).values for i in range(1, K+1)])
    log_q  = np.log1p(np.hstack([q_ask, q_bid])).astype(np.float32)
    spread = np.zeros(len(df_n), dtype=np.float32)   # INTC: ~constant 1-tick spread

    # Adding Trade Imbalance features (as computed in previous steps)
    ti_feats = np.column_stack([
        df_n['TI_20s'].values, df_n['TI_1min'].values, 
        df_n['TI_5min'].values, df_n['TI_15min'].values
    ]).astype(np.float32)

    # Continuous features vector: 2*K queues + 1 spread + 4 TIs = 15 features
    x_cont = np.hstack([log_q, spread[:, None], ti_feats])

    types   = df_n['type'].tolist()
    lvls    = df_n['lvl'].values.astype(int)
    hours   = df_n['hour'].values.astype(np.int64)
    lasts   = df_n['last_event_global'].values.astype(np.int64)

    # Global delta_t CLIPPED to [DT_MIN, DT_MAX] — prevents log(0) and extreme outliers
    dt_raw  = df_n['delta_time'].astype(float).values
    dt      = np.clip(dt_raw, DT_MIN, DT_MAX).astype(np.float32)

    sizes   = df_n['size'].fillna(1).astype(float).values.clip(1).astype(np.float32)
    
    # Map event (type, level) to integer label j in [0, 3*2*K - 1]
    event_j = np.array([encode_event_j(types[i], lvls[i], K)
                         for i in range(len(df_n))], dtype=np.int64)
    valid   = (event_j >= 0)   # Filter events with valid type/level mapping

    return dict(x_cont=x_cont, x_hour=hours, x_last=lasts,
                event_j=event_j, dt=dt, sizes=sizes, valid=valid)


print("Building MDQR dataset …")
ds_all = build_mdqr_dataset(df_norm, K)
N_all  = int(ds_all['valid'].sum())

# Random 80/20 split (standard strategy for temporal validation)
perm    = np.random.permutation(N_all)
N_train = int(0.8 * N_all)
valid_idx = np.where(ds_all['valid'])[0]
idx_tr  = valid_idx[perm[:N_train]]
idx_va  = valid_idx[perm[N_train:]]


def slice_ds(ds, idx):
    return {k: v[idx] for k, v in ds.items() if k != 'valid'}


ds_train = slice_ds(ds_all, idx_tr)
ds_val   = slice_ds(ds_all, idx_va)

# --- Diagnostics & Verification ---
print("-" * 60)
print(f"Train set: {N_train:,} events  |  Val set: {N_all - N_train:,} events")
print(f"Feature dimension (x_cont): {ds_train['x_cont'].shape[1]} (Expected: 15)")
print(f"Event distribution (train): "
      f"L={int((ds_train['event_j'] < 2*K).sum())}, "
      f"C={int(((ds_train['event_j'] >= 2*K) & (ds_train['event_j'] < 4*K)).sum())}, "
      f"M={int((ds_train['event_j'] >= 4*K).sum())}")
print(f"delta_t (clipped) — mean: {ds_train['dt'].mean()*1e3:.1f} ms  "
      f"median: {np.median(ds_train['dt'])*1e3:.1f} ms  "
      f"max: {ds_train['dt'].max():.1f} s")

# Encoding verification
j0 = encode_event_j('L', 1, K); j1 = encode_event_j('C', -3, K); j2 = encode_event_j('M', 1, K)
print(f"\nEncoding check:")
print(f"  L_ask1 = {j0} (expected 0)")
print(f"  C_bid3 = {j1} (expected {1*2*K + K + 3 - 1})")
print(f"  M_ask1 = {j2} (expected {2*2*K + 0})")
print("-" * 60)

Building MDQR dataset …
------------------------------------------------------------
Train set: 464,280 events  |  Val set: 116,071 events
Feature dimension (x_cont): 15 (Expected: 15)
Event distribution (train): L=226011, C=212347, M=25922
delta_t (clipped) — mean: 40.3 ms  median: 0.1 ms  max: 7.0 s

Encoding check:
  L_ask1 = 0 (expected 0)
  C_bid3 = 17 (expected 17)
  M_ask1 = 20 (expected 20)
------------------------------------------------------------


---
## 5. Training the Intensity Model

### 5.1 NLL loss (Eq. 4)

$$\mathcal{L}_\lambda = \frac{1}{|\mathcal{B}|}\sum_{k \in \mathcal{B}}
\Bigl[\Lambda(\mathbf{x}_k)\,\Delta t_k - \log\lambda^{(\eta_k,\ell_k)}(\mathbf{x}_k)\Bigr]$$

The clipped $\Delta t_k \in [1\,\text{ms}, 30\,\text{s}]$ ensures the survival term
$\Lambda \cdot \Delta t_k$ is always positive and meaningful, preventing the model from
collapsing to an infinite-intensity degenerate solution.

### 5.2 Optimizer and early stopping

**Optimizer:** Adam with cosine-annealing warm restarts (CAWR), oscillating between
$\eta_{\min}=10^{-5}$ and $\eta_{\max}=10^{-3}$.  The warm-restart schedule (period $T_0=50$
epochs, doubling at each restart) was carried over from Notebook 2: it periodically raises the
learning rate to escape plateaus — preferable to triangular schedules that can get stuck at
$\eta_{\min}$ for long stretches.  The scheduler is stepped **once per epoch**, as in NB2.

**Gradient clipping** at $\|\nabla\|_2 = 1.0$ prevents the survival term $\Lambda \cdot \Delta t_k$
from producing extremely large gradients at the start of training when $\Lambda$ is still near zero.

**Early stopping** with **patience = 30**: training halts when the validation NLL has not improved
by $\varepsilon = 10^{-6}$ for 30 consecutive epochs.  The paper uses patience = 10 on 3 months of
Bund futures (≈$3 \times 10^6$ events); with a single INTC trading day (≈50 k total events, ≈5 k
per level), the NLL curve is noisier and patience 30 is required to give $\Lambda$ time to converge,
matching the strategy used in Notebook 2.

In [ ]:
# ── Section 5 (MDQRNet Training) ────────────────────────────

def mdqr_nll(lambdas, event_j, dt):
    """
    Computes the Negative Log-Likelihood for the MDQR point process.
    Formula: mean( Lambda(x) * dt - log(lambda_j(x)) )
    
    Cast to float32 is enforced to prevent numerical instability under AMP.
    """
    # Force float32 for high-precision log calculations
    lambdas = lambdas.float()
    dt = dt.float()
    
    eps = 1e-9 # Safety epsilon to prevent log(0)
    
    # Global intensity (sum of all 30 conditional intensities)
    Lambda = lambdas.sum(dim=1)
    
    # Intensity of the specific event that occurred
    idx = event_j.clamp(0).unsqueeze(1)
    selected_lambda = lambdas.gather(1, idx).squeeze(1)
    
    # Term 1: Integrated intensity over the interval dt (Survival term)
    survival_term = Lambda * dt
    
    # Term 2: Log-intensity of the observed event (Marked term)
    log_term = torch.log(selected_lambda + eps)
    
    return (survival_term - log_term).mean()

print(np.isnan(ds_train['x_cont']).any()) # Doit être False

def _to_tensors(ds):
    return (torch.FloatTensor(ds['x_cont']),
            torch.LongTensor(ds['x_hour']),
            torch.LongTensor(ds['x_last']),
            torch.LongTensor(ds['event_j']),
            torch.FloatTensor(ds['dt']))

Xc_tr, Xh_tr, Xl_tr, Ej_tr, Dt_tr = _to_tensors(ds_train)
Xc_va, Xh_va, Xl_va, Ej_va, Dt_va = _to_tensors(ds_val)

BATCH    = 4096
EPOCHS   = 300   
LR_MIN   = 1e-5
LR_MAX   = 1e-3
T0       = 50
PATIENCE = 30    

# Optimized DataLoader
# Note: If a 'BrokenPipeError' occurs on Windows/macOS, set num_workers=0.
loader_tr = DataLoader(
    TensorDataset(Xc_tr, Xh_tr, Xl_tr, Ej_tr, Dt_tr),
    batch_size=BATCH, 
    shuffle=True, 
    drop_last=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

use_cuda = device.type == 'cuda'

# Initialize network with n_cont=15 (11 baseline + 4 Trade Imbalance features)
mdqr_net = MDQRNet(K=K, n_cont=15, n_hour=7, n_last=7).to(device)

# WARNING: torch.compile is intentionally disabled for Windows compatibility.
# If running on Linux/WSL2, this block can be uncommented.
# if hasattr(torch, 'compile') and use_cuda:
#     try:
#         mdqr_net = torch.compile(mdqr_net)
#         print("MDQRNet compiled successfully.")
#     except Exception:
#         pass

opt_int   = optim.Adam(mdqr_net.parameters(), lr=LR_MAX, weight_decay=1e-4)
sched_int = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt_int, T_0=T0, T_mult=2, eta_min=LR_MIN)

# Initialize Gradient Scaler for Automatic Mixed Precision (CUDA only)
scaler = torch.cuda.amp.GradScaler() if use_cuda else None

train_nll, val_nll = [], []
best_val, best_state, no_imp = np.inf, None, 0

print(f"Training MDQRNet: up to {EPOCHS} epochs, batch={BATCH}, "
      f"LR [{LR_MIN:.0e},{LR_MAX:.0e}] CAWR (T0={T0}) ...")

for epoch in range(1, EPOCHS + 1):
    mdqr_net.train()
    ep_loss = 0.0
    for xc, xh, xl, ej, dt in loader_tr:
        xc, xh, xl, ej, dt = (t.to(device) for t in (xc, xh, xl, ej, dt))
        opt_int.zero_grad()
        
        if use_cuda:
            with torch.autocast(device_type='cuda'):
                loss = mdqr_nll(mdqr_net(xc, xh, xl), ej, dt)
            scaler.scale(loss).backward()
            scaler.unscale_(opt_int) 
            nn.utils.clip_grad_norm_(mdqr_net.parameters(), 1.0)
            scaler.step(opt_int)
            scaler.update()
        else:
            loss = mdqr_nll(mdqr_net(xc, xh, xl), ej, dt)
            loss.backward()
            nn.utils.clip_grad_norm_(mdqr_net.parameters(), 1.0)
            opt_int.step()
            
        ep_loss += loss.item() * len(xc)
        
    sched_int.step()
    ep_loss /= len(Xc_tr)

    mdqr_net.eval()
    with torch.no_grad():
        va_parts = []
        for i in range(0, len(Xc_va), BATCH):
            s = slice(i, i + BATCH)
            l = mdqr_nll(mdqr_net(Xc_va[s].to(device), Xh_va[s].to(device), Xl_va[s].to(device)),
                         Ej_va[s].to(device), Dt_va[s].to(device))
            va_parts.append(l.item() * len(Xc_va[s]))
        val_loss = sum(va_parts) / len(Xc_va)

    train_nll.append(ep_loss)
    val_nll.append(val_loss)

    if val_loss < best_val - 1e-6:
        best_val   = val_loss
        best_state = {k: v.cpu().clone() for k, v in mdqr_net.state_dict().items()}
        no_imp     = 0
    else:
        no_imp += 1

    if epoch % 20 == 0 or epoch == 1:
        lr_now = opt_int.param_groups[0]['lr']
        print(f"  Epoch {epoch:3d}/{EPOCHS}: train NLL={ep_loss:.4f}  val NLL={val_loss:.4f}"
              f"  LR={lr_now:.2e}  patience={PATIENCE-no_imp}/{PATIENCE}")
    if no_imp >= PATIENCE:
        print(f"  Early stop at epoch {epoch}  (best val={best_val:.4f})")
        break

mdqr_net.load_state_dict(best_state)
print(f"\nFinal best val NLL: {best_val:.4f}  (trained {len(train_nll)} epochs)")

False
Training MDQRNet: up to 300 epochs, batch=4096, LR [1e-05,1e-03] CAWR (T0=50) ...
  Epoch   1/300: train NLL=inf  val NLL=0.6191  LR=9.99e-04  patience=30/30
  Epoch  20/300: train NLL=inf  val NLL=-0.1458  LR=6.58e-04  patience=30/30
  Epoch  40/300: train NLL=inf  val NLL=-0.2284  LR=1.05e-04  patience=30/30


KeyboardInterrupt: 

### Figure 5 — Learning curve (cf. Figure 5 of the paper)

The negative log-likelihood is expected to decrease monotonically on the training set and
converge on the validation set.  The inset (if available) zooms in on the final epochs to
show convergence behaviour.  A gap between train and val NLL reflects the limited size of
the dataset (one trading day vs three months in the paper).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
epochs_done = len(train_nll)
xs = np.arange(1, epochs_done + 1)
ax.plot(xs, train_nll, color='#F44336', lw=2, label='Train NLL')
ax.plot(xs, val_nll,   color='#2196F3', lw=2, label='Val NLL')
ax.axhline(best_val, ls='--', lw=1, color='gray',
           label=f'Best val = {best_val:.4f}')
best_ep = int(np.argmin(val_nll)) + 1
ax.axvline(best_ep, ls=':', lw=1, color='gray')
ax.set_xlabel('Epoch'); ax.set_ylabel('NLL')
ax.set_title('Figure 5 — MDQR Intensity Model: Learning Curve')
ax.legend(); plt.tight_layout()
plt.savefig('fig05_learning_curve.pdf', dpi=150); plt.show()

# ── diagnostic output ──────────────────────────────────────────────────────
print("=" * 60)
print("DIAGNOSTIC — Fig 5 (learning curve)")
print(f"  Epochs trained:   {epochs_done}")
print(f"  Best epoch:       {best_ep}")
print(f"  Best val NLL:     {best_val:.4f}")
print(f"  Final train NLL:  {train_nll[-1]:.4f}")
print(f"  Val NLL at ep 1:  {val_nll[0]:.4f}")
print(f"  Val NLL at ep {epochs_done}: {val_nll[-1]:.4f}")
print(f"  Monotone decrease in val? {all(val_nll[i]>=val_nll[i+1] for i in range(min(10,len(val_nll)-1)))}")
print("=" * 60)

In [ ]:
# ── Order size distribution analysis and strategy decision ──────────
sizes = df_norm['size'].dropna().values
max_size = sizes.max()
p99 = np.percentile(sizes, 99)
pct_under_200 = (sizes <= 200).mean() * 100

print(f"Normalized order size statistics (INTC):")
print(f"  Maximum observed size: {max_size:.0f} shares/lots")
print(f"  99th percentile: {p99:.0f}")
print(f"  % of orders <= 200: {pct_under_200:.2f}%\n")

if p99 <= 200:
    print("-> STRATEGY DECISION: Use Exact Integer Classes (like the paper).")
else:
    print("-> STRATEGY DECISION: Use Quantile Bins (original approach is better for INTC).")

plt.figure(figsize=(8, 4))
plt.hist(sizes[sizes <= p99], bins=50, color='#9C27B0', alpha=0.7)
plt.title(f"Distribution of order sizes up to 99th percentile ({p99:.0f})")
plt.xlabel("Size")
plt.ylabel("Frequency")
plt.grid(alpha=0.3)
plt.show()

---
## 6. Training the Order-Size Model (SizeNet)

### 6.1 Discretisation (Eq. 5)

Order sizes $s_k$ (AES-normalised) are discretised into $C$ **quantile bins**:
the bin edges $b_0 < b_1 < \cdots < b_C$ are chosen so that each bin $[b_{c-1}, b_c)$ contains
roughly $1/C$ of the training events.  Quantile binning is preferable to equal-width bins for
heavy-tailed LOB size distributions: it guarantees balanced class frequencies, preventing the
cross-entropy from collapsing to the majority class.

The SizeNet loss is

$$\mathcal{L}_s(\phi) = -\frac{1}{N}\sum_{k=1}^N \log \hat{p}_{c_k}(s_k \mid \eta_k, \ell_k, \mathbf{x}_k;\phi)$$

where $c_k = \arg\min_c \{b_c \geq s_k\}$ is the bin index of $s_k$.

### 6.2 Conditioning

SizeNet takes $(\mathbf{x}_k,\, j_k)$ as input, where $j_k \in \{0,\ldots,29\}$ is the event
index embedded into a 2-dimensional vector.  This allows the model to learn a *separate size
profile* for each (type, level, side) combination — market orders at level 1 tend to be larger
than cancellations at level 5, for instance.

### 6.3 Training

Same CAWR optimizer schedule as the intensity model; same patience = 30 early stopping.

In [ ]:
# ── Section 6 (SizeNet Training) ────────────────────────────

sz_loader = DataLoader(
    TensorDataset(Xc_s_tr, Xh_s_tr, Xl_s_tr, Xj_s_tr, Ys_tr),
    batch_size=BATCH, 
    shuffle=True, 
    drop_last=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

use_cuda = device.type == 'cuda'

# Initialize network with n_cont=15
size_net = SizeNet(K=K, n_cont=15, n_hour=7, n_last=7,
                   n_event=3*2*K, n_classes=N_SIZE_CLS).to(device)

# WARNING: torch.compile is intentionally disabled for Windows compatibility.
# If running on Linux/WSL2, this block can be uncommented.
# if hasattr(torch, 'compile') and use_cuda:
#     try:
#         mdqr_net = torch.compile(mdqr_net)
#         print("MDQRNet compiled successfully.")
#     except Exception:
#         pass

opt_sz   = optim.Adam(size_net.parameters(), lr=LR_MAX, weight_decay=1e-4)
sched_sz = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt_sz, T_0=T0, T_mult=2, eta_min=LR_MIN)
ce_fn    = nn.CrossEntropyLoss()

scaler_sz = torch.cuda.amp.GradScaler() if use_cuda else None

SZ_EPOCHS = 100
best_sz, best_sz_state, no_imp_sz = np.inf, None, 0

print(f"Training SizeNet: up to {SZ_EPOCHS} epochs, batch={BATCH} ...")
for epoch in range(1, SZ_EPOCHS + 1):
    size_net.train()
    ep_sz = 0.0
    for xc, xh, xl, xj, ys in sz_loader:
        xc, xh, xl, xj, ys = (t.to(device) for t in (xc, xh, xl, xj, ys))
        opt_sz.zero_grad()
        
        if use_cuda:
            with torch.autocast(device_type='cuda'):
                probs = size_net(xc, xh, xl, xj)
                loss  = ce_fn(torch.log(probs + 1e-10), ys)
            scaler_sz.scale(loss).backward()
            scaler_sz.step(opt_sz)
            scaler_sz.update()
        else:
            probs = size_net(xc, xh, xl, xj)
            loss  = ce_fn(torch.log(probs + 1e-10), ys)
            loss.backward()
            opt_sz.step()
            
        ep_sz += loss.item() * len(xc)
        
    sched_sz.step()
    ep_sz /= len(Xc_s_tr)
    
    size_net.eval()
    with torch.no_grad():
        pv_parts = []
        for i in range(0, len(Xc_s_va), BATCH):
            s = slice(i, i + BATCH)
            pv = size_net(Xc_s_va[s].to(device), Xh_s_va[s].to(device),
                           Xl_s_va[s].to(device), Xj_s_va[s].to(device))
            pv_parts.append(ce_fn(torch.log(pv+1e-10), Ys_va[s].to(device)).item() * len(pv))
        vl = sum(pv_parts) / len(Xc_s_va)
        
    if vl < best_sz - 1e-6:
        best_sz = vl
        best_sz_state = {k: v.cpu().clone() for k, v in size_net.state_dict().items()}
        no_imp_sz = 0
    else:
        no_imp_sz += 1
        
    if epoch % 20 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}: train CE={ep_sz:.4f}  val CE={vl:.4f}")
    if no_imp_sz >= PATIENCE:
        print(f"  Early stop at epoch {epoch}")
        break

size_net.load_state_dict(best_sz_state)
print(f"Best SizeNet val CE: {best_sz:.4f}")

---
## 7. Model Evaluation

We evaluate the intensity model on the held-out validation set using three complementary metrics
(cf. Section 3.4.3 of the paper and Notebook 2, Section 6):

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Log-likelihood** | $-\mathcal{L}_\lambda$ per event | Overall fit; combines timing and type prediction (higher = better) |
| **Balanced accuracy** | Macro-averaged recall over the 30 event-level classes | Directional accuracy of the predicted argmax event (higher = better) |
| **Timing relative error** | $\text{median}_k \frac{|\hat{\Delta t}_k - \Delta t_k|}{\Delta t_k} \times 100\%$ | Inter-arrival calibration, where $\hat{\Delta t}_k = 1/\Lambda(\mathbf{x}_k)$ (lower = better) |

**LOBSTER batch-arrival filter for the timing metric (identical to NB2).** The raw $\Delta t_k$
distribution is dominated by near-zero values (LOBSTER batch arrivals, $\Delta t \approx 0$).
Clipping restores these to $1\,\text{ms}$, but the corresponding $|1/\Lambda - 1\,\text{ms}| / 1\,\text{ms}$
ratios are still uninformative (the model's predicted inter-arrival $1/\Lambda \gg 1\,\text{ms}$
for any reasonable $\Lambda$, inflating the metric to thousands of percent).  Following NB2,
**the timing metric is evaluated only on events with the original (unclipped) $\Delta t_k > 1\,\text{ms}$**
— the genuine independent inter-arrivals that approximate the Poisson assumption.

**Theoretical floor $\approx 44\%$.** For a perfectly calibrated homogeneous Poisson process
with rate $\lambda$, the predicted inter-arrival is $1/\lambda$ and the true inter-arrival is
$T \sim \text{Exp}(\lambda)$.  The median of $T$ is $\ln 2 / \lambda$, so even a perfect model
achieves a median relative error of $|1/\lambda - \ln 2/\lambda| / (\ln 2/\lambda) = 1/\ln 2 - 1 \approx 44\%$.
Any timing metric above this floor reflects the residual randomness intrinsic to the Poisson model,
not estimation error.

In [ ]:
def evaluate_mdqr(net, ds, K, device):
    from sklearn.metrics import balanced_accuracy_score
    net.eval()
    xc = torch.FloatTensor(ds['x_cont'])
    xh = torch.LongTensor(ds['x_hour'])
    xl = torch.LongTensor(ds['x_last'])
    ej = ds['event_j']
    dt = ds['dt']  # already clipped

    lam = []
    with torch.no_grad():
        for i in range(0, len(xc), BATCH):
            s = slice(i, i + BATCH)
            lam.append(net(xc[s].to(device), xh[s].to(device), xl[s].to(device)).cpu())
    lam = torch.cat(lam, 0).numpy()

    Lambda  = lam.sum(axis=1)
    pred_dt = 1.0 / (Lambda + 1e-12)
    pred_j  = lam.argmax(axis=1)

    log_lj  = np.log(lam[np.arange(len(ej)), ej] + 1e-12)
    nll_val = (Lambda * dt - log_lj).mean()

    try:    bal_acc = balanced_accuracy_score(ej, pred_j)
    except: bal_acc = np.nan

    # Timing on ORIGINAL (unclipped) delta_t, filtered to genuine inter-arrivals (DT_EVAL)
    dt_orig = df_norm['delta_time'].astype(float).values[
        np.where(ds_all['valid'])[0][perm[N_train:]]
    ]
    tm = dt_orig > DT_EVAL
    if tm.sum() > 0:
        rel       = np.abs(pred_dt[tm] - dt_orig[tm]) / (dt_orig[tm] + 1e-12)
        timing_pct = np.median(rel) * 100.0
    else:
        timing_pct = np.nan

    return dict(log_lik=-nll_val, bal_acc=bal_acc,
                timing_pct=timing_pct, n_timing=int(tm.sum()),
                Lambda_mean=float(Lambda.mean()), Lambda_std=float(Lambda.std()),
                pred_dt_mean_ms=float(pred_dt.mean()*1e3),
                pred_dt_median_ms=float(np.median(pred_dt)*1e3))


print("Evaluating MDQR on validation set …")
res = evaluate_mdqr(mdqr_net, ds_val, K, device)

print("=" * 60)
print("DIAGNOSTIC — Section 7 (evaluation)")
for k, v in res.items():
    if isinstance(v, float):
        print(f"  {k:<28}: {v:.4f}")
    else:
        print(f"  {k:<28}: {v}")
print("=" * 60)

---
## 8. LOB Simulation via the Gillespie Algorithm

Given a trained MDQR model, synthetic LOB paths are generated by the exact **Gillespie algorithm**
for a continuous-time multidimensional Poisson process:

$$
\text{while } t < T_{\text{sim}}:
\quad
\Delta t \sim \text{Exp}\!\left(\frac{1}{\Lambda(\mathbf{x})}\right),
\quad
j \sim \text{Categorical}\!\left(\frac{\lambda_j(\mathbf{x})}{\Lambda(\mathbf{x})}\right),
\quad
s \sim \text{SizeNet}(\mathbf{x}, j)
$$

**Queue update rules** (applied after each sampled event):

| Event type | Affected queue | Update |
|------------|---------------|--------|
| Limit ($L$) at level $\ell$, side $\sigma$ | $Q^\sigma_\ell$ | $\min(Q^\sigma_\ell + s,\; Q_{\max})$ |
| Cancel ($C$) or Market ($M$) at level $\ell$, side $\sigma$ | $Q^\sigma_\ell$ | $\max(Q^\sigma_\ell - s,\; 0)$ |

If a best-quote queue ($\ell = 1$) reaches 0 after a market order, the mid-price shifts
$\pm 1$ tick and the queue vector rolls inward (level 2 becomes level 1, etc.).

$Q_{\max} = 50$ (normalised) caps queue growth to prevent unbounded state accumulation
in long simulations.  The simulator is initialised from the last observed LOB state.

In [ ]:
# ──Simulation (Section 8) ───────────────────────

Q_MAX = 50  # cap normalised queue size to prevent explosion

def simulate_mdqr(mdqr_net, size_net, init_q, K, sz_bins,
                  T_sim=23400.0, t_start=MARKET_OPEN_S,
                  max_events=2_000_000, device='cpu'):
    mdqr_net.eval(); size_net.eval()
    q         = np.array(init_q, dtype=float).clip(0, Q_MAX)
    t         = 0.0
    hour      = 0
    last_enc  = 0
    mid_ticks = 0
    events    = []
    t_list    = [0.0]
    
    # Buffer for recent trades (t, vol_buy, vol_sell)
    recent_trades = []

    with torch.no_grad():
        while t < T_sim and len(events) < max_events:
            # Clean up trades older than 15 mins (900s)
            while recent_trades and recent_trades[0][0] < t - 900:
                recent_trades.pop(0)
            
            # On-the-fly calculation of Trade Imbalances
            ti_feats = [0.0, 0.0, 0.0, 0.0]
            if recent_trades:
                arr = np.array(recent_trades)
                for i, tau in enumerate([20, 60, 300, 900]):
                    mask = arr[:, 0] >= t - tau
                    if np.any(mask):
                        v_b, v_s = np.sum(arr[mask, 1]), np.sum(arr[mask, 2])
                        tot = v_b + v_s
                        ti_feats[i] = (v_b - v_s) / tot if tot > 0 else 0.0
            
            # Updated x_cont vector (2*K queues + 1 spread + 4 TIs = 15 features)
            x_cont = np.concatenate([np.log1p(q), [0.0], ti_feats]).astype(np.float32)
            xc = torch.FloatTensor(x_cont).unsqueeze(0).to(device)
            xh = torch.LongTensor([min(hour, 6)]).to(device)
            xl = torch.LongTensor([last_enc]).to(device)

            lam    = mdqr_net(xc, xh, xl).squeeze(0).cpu().numpy().clip(1e-10)
            Lambda = lam.sum()

            dt_sim = np.random.exponential(1.0 / Lambda)
            t     += dt_sim
            if t > T_sim:
                break
            t_list.append(t)

            j  = np.random.choice(len(lam), p=lam / Lambda)
            ti_idx, si, li = decode_event_j(j, K)
            level = li + 1

            # Sample size using the quantile bins
            if sz_bins is not None:
                xj = torch.LongTensor([j]).to(device)
                sp = size_net(xc, xh, xl, xj).squeeze(0).cpu().numpy()
                sc = np.random.choice(len(sp), p=sp)
                size = max(1, int(0.5 * (sz_bins[sc] + sz_bins[sc + 1])))
            else:
                size = 1

            last_enc = ti_idx * 2 + si + 1   # 1..6
            events.append((t, ti_idx, si, level, size, last_enc))

            # Update the trades buffer if it's a Market Order (ti_idx == 2)
            if ti_idx == 2:
                v_b = size if si == 0 else 0.0
                v_s = size if si == 1 else 0.0
                recent_trades.append((t, v_b, v_s))

            # Queue update
            qa = q[:K].copy(); qb = q[K:].copy()
            if ti_idx == 0:   # Limit
                if si == 0: qa[li] = min(qa[li] + size, Q_MAX)
                else:       qb[li] = min(qb[li] + size, Q_MAX)
            else:             # Cancel or Market
                if si == 0: qa[li] = max(0.0, qa[li] - size)
                else:       qb[li] = max(0.0, qb[li] - size)
                # price shift when level-1 depletes
                if li == 0:
                    if si == 0 and qa[0] == 0:
                        mid_ticks += 1
                        qa = np.roll(qa, -1); qa[-1] = max(1.0, qb[0] * 0.5)
                    elif si == 1 and qb[0] == 0:
                        mid_ticks -= 1
                        qb = np.roll(qb, -1); qb[-1] = max(1.0, qa[0] * 0.5)
            q    = np.concatenate([qa, qb])
            hour = min(int((t_start + t) / 3600) - int(t_start / 3600), 6)

    return events, np.array(t_list), mid_ticks


# ── initialise from last training row ─────────────────────────────────────
last_row = df_norm.iloc[-1]
init_q   = [max(1.0, float(last_row.get(f'Q_{i}',  1))) for i in range(1, K+1)] + \
           [max(1.0, float(last_row.get(f'Q_-{i}', 1))) for i in range(1, K+1)]

print("Running MDQR simulation (one full trading day) …")
sim_events, sim_times, sim_mid = simulate_mdqr(
    mdqr_net, size_net, init_q, K,
    T_sim=23400.0, sz_bins=sz_bins, device=device
)
print(f"Simulated {len(sim_events):,} events  |  "
      f"duration {sim_times[-1]/3600:.2f} h  |  mid shift {sim_mid:+d} ticks")

_TYPES = ['L', 'C', 'M']
df_sim = pd.DataFrame([
    {'time': t, 'type': _TYPES[ti_idx],
     'lvl': lv if si == 0 else -lv, 'size': sz}
    for (t, ti_idx, si, lv, sz, _) in sim_events
])
df_sim['delta_time'] = df_sim['time'].diff().fillna(0)

print("=" * 60)
print("DIAGNOSTIC — Section 8 (simulation)")
print(f"  Total events:    {len(sim_events):,}")
print(f"  Events/second:   {len(sim_events)/sim_times[-1]:.1f}")
print(f"  Simulated hours: {sim_times[-1]/3600:.3f}")
print("  Type distribution (sim):")
for t in ['L','C','M']:
    n = (df_sim['type']==t).sum()
    print(f"    {t}: {n:,} ({n/len(df_sim)*100:.1f}%)")
print("  Type distribution (real):")
for t in ['L','C','M']:
    n = (df_norm['type']==t).sum()
    print(f"    {t}: {n:,} ({n/len(df_norm)*100:.1f}%)")
print("=" * 60)

---
## 9. Validation — Figure 9: Event Transition Matrix

The empirical transition matrix $T[i,j] = \hat{P}(\eta_{k+1} = j \mid \eta_k = i)$ is
a direct measure of *excitation* between event types.  In the historical data, strong
diagonal dominance is observed: a cancellation tends to be followed by another cancellation
(cancel→cancel $\approx 0.73$ in the paper), and a trade tends to trigger further trades.
The QR model, by construction, produces uniform rows.

The MDQR model incorporates $\eta_{k-1}$ via the `last_event_global` embedding, so it should
reproduce at least part of this excitation structure — though the degree of reproduction depends
on how much excitation information is preserved through the single shared global embedding
(as opposed to per-level embeddings as in the full paper).

In [ ]:
def compute_3x3_transition(df, type_col='type'):
    cats = df[type_col].map(EVENT_TO_IDX).dropna().values.astype(int)
    T = np.zeros((3, 3))
    np.add.at(T, (cats[:-1], cats[1:]), 1)
    rs = T.sum(axis=1, keepdims=True); rs[rs == 0] = 1
    return T / rs

def _plot_transition(T, title, ax, vmax=0.85):
    labs = ['Limit', 'Cancel', 'Market']
    im = ax.imshow(T, cmap='YlOrRd', vmin=0, vmax=vmax, aspect='equal')
    ax.set_xticks(range(3)); ax.set_xticklabels(labs, rotation=30, ha='right', fontsize=10)
    ax.set_yticks(range(3)); ax.set_yticklabels(labs, fontsize=10)
    ax.set_xlabel('Next event', fontsize=10); ax.set_ylabel('Current event', fontsize=10)
    for r in range(3):
        for c in range(3):
            ax.text(c, r, f'{T[r,c]:.2f}', ha='center', va='center', fontsize=11,
                    color='white' if T[r,c] > 0.5 else 'black')
    ax.set_title(title, fontsize=12)
    return im

T_real = compute_3x3_transition(df_norm)
T_mdqr = compute_3x3_transition(df_sim)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im = _plot_transition(T_real, 'Historical data', axes[0])
_plot_transition(T_mdqr, 'MDQR simulation', axes[1])
plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75, label='Transition probability')
plt.suptitle('Figure 9 — Event Transition Matrix', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('fig09_transition.pdf', dpi=150); plt.show()

print("=" * 60)
print("DIAGNOSTIC — Fig 9 (transition matrix)")
labs = ['L','C','M']
print("Real:")
for i in range(3):
    print(f"  {labs[i]} -> {dict(zip(labs, T_real[i].round(3)))}")
print("MDQR simulation:")
for i in range(3):
    print(f"  {labs[i]} -> {dict(zip(labs, T_mdqr[i].round(3)))}")
print(f"Cancel->Cancel: real={T_real[1,1]:.3f}  MDQR={T_mdqr[1,1]:.3f}  (paper: ~0.73)")
print(f"Trade->Trade:   real={T_real[2,2]:.3f}  MDQR={T_mdqr[2,2]:.3f}  (paper: ~0.30)")
print("=" * 60)

---
## 10. Validation — Figures 10–11: Queue Size Distributions and LOB Profile

### Figure 10 — Queue size distribution at ask level 1

Empirical LOB queue sizes are well-described by a **Gamma distribution** after AES
normalisation (Bodor & Carlier 2024).  The Q–Q plot provides a non-parametric comparison
of the simulated and historical quantile functions.

### Figure 11 — Average LOB profile

The mean normalised queue size $\bar{Q}_i$ should decrease with level distance from
the best quote (i.e., $\bar{Q}_1 > \bar{Q}_2 > \cdots$), reflecting the higher activity
near the top of the book.  The MDQR simulator should reproduce this shape because the
queue-update rules (limit, cancel, market) are applied per-level.

In [ ]:
def reconstruct_q_matrix(events, init_q, K, max_ev=300_000):
    # Re-apply queue updates from the simulation to get queue state at each step
    q    = np.array(init_q, dtype=float)
    rows = []
    for ev in events[:max_ev]:
        t, ti, si, lv, sz, _ = ev
        li = lv - 1
        qa, qb = q[:K].copy(), q[K:].copy()
        if ti == 0:
            if si == 0: qa[li] = min(qa[li] + sz, Q_MAX)
            else:       qb[li] = min(qb[li] + sz, Q_MAX)
        else:
            if si == 0: qa[li] = max(0.0, qa[li] - sz)
            else:       qb[li] = max(0.0, qb[li] - sz)
        q = np.concatenate([qa, qb])
        rows.append(q.copy())
    return np.array(rows)

sim_q_mat = reconstruct_q_matrix(sim_events, init_q, K)
real_q1   = df_norm['Q_1'].fillna(0).astype(float).values
real_q1   = real_q1[real_q1 > 0]
sim_q1    = sim_q_mat[:, 0]; sim_q1 = sim_q1[sim_q1 > 0]

# ── Figure 10: histogram + gamma fit + Q-Q ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
mx = float(np.percentile(real_q1, 95))
bins = np.linspace(0, mx, 50)

ax = axes[0]
ax.hist(real_q1, bins=bins, density=True, alpha=0.6, color='#2196F3', label='Historical')
ax.hist(sim_q1,  bins=bins, density=True, alpha=0.6, color='#F44336', label='MDQR')
a_, _, sc_ = gamma_dist.fit(real_q1, floc=0)
x_f = np.linspace(0, mx, 300)
ax.plot(x_f, gamma_dist.pdf(x_f, a_, 0, sc_), '--', color='#2196F3', lw=2,
        label=f'Gamma fit (α={a_:.2f})')
ax.set_xlabel('Normalised queue size'); ax.set_ylabel('Density')
ax.set_title('Ask Level 1 — Queue Distribution'); ax.legend(fontsize=9)

ax = axes[1]
qs = np.linspace(0.01, 0.99, 100)
qr, qs_s = np.quantile(real_q1, qs), np.quantile(sim_q1, qs)
ax.plot(qr, qs_s, '.', color='#F44336', ms=4)
d = [0, max(qr[-1], qs_s[-1])]
ax.plot(d, d, 'k--', lw=1, label='$y=x$')
ax.set_xlabel('Historical quantiles'); ax.set_ylabel('MDQR quantiles')
ax.set_title('Q–Q Plot: Ask Level 1'); ax.legend(fontsize=9)

plt.suptitle('Figure 10 — Queue Size Distribution (Ask Level 1)', fontweight='bold')
plt.tight_layout(); plt.savefig('fig10_queue_dist.pdf', dpi=150); plt.show()

print("=" * 60)
print("DIAGNOSTIC — Fig 10 (queue distribution, ask L1)")
print(f"  Historical: mean={real_q1.mean():.3f}  std={real_q1.std():.3f}  "
      f"median={np.median(real_q1):.3f}  p95={np.percentile(real_q1,95):.3f}")
print(f"  MDQR sim:   mean={sim_q1.mean():.3f}  std={sim_q1.std():.3f}  "
      f"median={np.median(sim_q1):.3f}  p95={np.percentile(sim_q1,95):.3f}")
print(f"  Gamma fit (real): alpha={a_:.3f}  scale={sc_:.3f}")
_qq_corr = np.corrcoef(qr, qs_s)[0,1]
print(f"  Q-Q correlation: {_qq_corr:.4f}  (1.0 = perfect match)")
print("=" * 60)

# ── Figure 11: average LOB profile ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, label, use_real in [(axes[0],'Historical',True),(axes[1],'MDQR Simulation',False)]:
    ask_m = []; bid_m = []
    for i in range(1, K+1):
        if use_real:
            ask_m.append(df_norm[f'Q_{i}'].fillna(0).astype(float).mean())
            bid_m.append(df_norm[f'Q_-{i}'].fillna(0).astype(float).mean())
        else:
            ask_m.append(sim_q_mat[:, i-1].mean())
            bid_m.append(sim_q_mat[:, K+i-1].mean())
    lvs = np.arange(1, K+1)
    ax.bar(lvs - 0.2, ask_m, 0.35, color='#F44336', alpha=0.8, label='Ask')
    ax.bar(lvs + 0.2, bid_m, 0.35, color='#2196F3', alpha=0.8, label='Bid')
    ax.set_xlabel('Level'); ax.set_ylabel('Mean normalised queue size')
    ax.set_title(label); ax.legend()
plt.suptitle('Figure 11 — Average LOB Profile', fontweight='bold')
plt.tight_layout(); plt.savefig('fig11_lob_profile.pdf', dpi=150); plt.show()

print("=" * 60)
print("DIAGNOSTIC — Fig 11 (LOB profile)")
for i in range(1, K+1):
    rA = df_norm[f'Q_{i}'].fillna(0).astype(float).mean()
    rB = df_norm[f'Q_-{i}'].fillna(0).astype(float).mean()
    sA = sim_q_mat[:, i-1].mean()
    sB = sim_q_mat[:, K+i-1].mean()
    print(f"  Level {i}: ask real={rA:.3f} sim={sA:.3f} | bid real={rB:.3f} sim={sB:.3f}")
print("=" * 60)

---
## 11. Validation — Figure 12: Cross-Level Queue-Size Correlation Matrix

The $10 \times 10$ Pearson correlation matrix captures linear dependencies between queue
sizes across all levels.  The ordering — bid 1, ask 1, bid 2, ask 2, …, bid 5, ask 5 —
interleaves bid and ask queues to highlight the anti-correlation between opposing best queues.

The paper reports a strong **negative** bid1–ask1 correlation ($\approx -0.54$), reflecting
the mean-reverting nature of the bid-ask spread: a large bid queue tends to precede a trade
that depletes the ask queue (or vice versa).  The MDQR model should capture at least part of
this structure because queue states at all levels are included in $\mathbf{x}_k$.

In [ ]:
labels_2K = []
for i in range(1, K+1):
    labels_2K += [f'B{i}', f'A{i}']

def build_corr_matrix(df_n=None, q_mat=None, K=5):
    cols = []
    for i in range(1, K+1):
        if df_n is not None:
            cols.append(df_n[f'Q_-{i}'].fillna(0).astype(float).values)
            cols.append(df_n[f'Q_{i}'].fillna(0).astype(float).values)
        else:
            cols.append(q_mat[:, K+i-1])
            cols.append(q_mat[:, i-1])
    return np.corrcoef(np.column_stack(cols).T)

C_real = build_corr_matrix(df_n=df_norm, K=K)
C_sim  = build_corr_matrix(q_mat=sim_q_mat, K=K)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, C, title in [(axes[0], C_real, 'Historical'), (axes[1], C_sim, 'MDQR Simulation')]:
    im = ax.imshow(C, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(2*K)); ax.set_xticklabels(labels_2K, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(2*K)); ax.set_yticklabels(labels_2K, fontsize=8)
    for r in range(2*K):
        for c in range(2*K):
            ax.text(c, r, f'{C[r,c]:.2f}', ha='center', va='center', fontsize=6,
                    color='white' if abs(C[r,c]) > 0.6 else 'black')
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Figure 12 — Cross-Level Queue-Size Correlation Matrix', fontweight='bold')
plt.tight_layout(); plt.savefig('fig12_corr_matrix.pdf', dpi=150); plt.show()

print("=" * 60)
print("DIAGNOSTIC — Fig 12 (correlation matrix)")
print(f"  Real  bid1–ask1 (B1–A1): {C_real[0,1]:.3f}  (paper: ~-0.54)")
print(f"  MDQR  bid1–ask1 (B1–A1): {C_sim[0,1]:.3f}")
print(f"  Real  ask1–ask2 (A1–A2): {C_real[1,3]:.3f}")
print(f"  MDQR  ask1–ask2 (A1–A2): {C_sim[1,3]:.3f}")
# Frobenius distance between matrices
frob = np.linalg.norm(C_real - C_sim, 'fro')
print(f"  Frobenius distance real vs MDQR: {frob:.4f}")
print("=" * 60)

---
## 12. Validation — Figure 13: 1-Minute Log-Return Distribution

We reconstruct a mid-price path from the simulated event sequence by tracking tick-level
shifts whenever the level-1 queue is depleted by a trade.  The resulting 1-minute returns
should exhibit the well-known stylised fact of **excess kurtosis** (heavy tails relative to
the Gaussian), quantified by the Q–Q plot against $\mathcal{N}(0,1)$.

Note that the mid-price process in our simplified simulator does not include all price-discovery
mechanisms (e.g., limit orders improving the best quote), so the return volatility may differ
from the historical value.  The shape of the distribution — specifically the departure from
Gaussianity — is the primary target.

In [ ]:
def compute_mid_path(events):
    times, prices = [0.0], [0.0]
    mid = 0.0
    for t, ti, si, lv, sz, _ in events:
        if ti == 2 and lv == 1:
            mid += 1 if si == 0 else -1
        times.append(t)
        prices.append(mid)
    return np.array(times), np.array(prices)

def resample_returns(times, prices, freq=60.0):
    if times[-1] < freq:
        return np.array([])
    grid   = np.arange(0, times[-1], freq)
    px_g   = np.interp(grid, times, prices)
    return np.diff(px_g)

times_sim, px_sim = compute_mid_path(sim_events)
ret_sim  = resample_returns(times_sim, px_sim, freq=60.0)

px_hist  = 0.5 * (ob['ask_px_1'].values + ob['bid_px_1'].values).astype(float)
t_hist   = msg['time'].values.astype(float)
grid_h   = np.arange(t_hist.min(), t_hist.max(), 60.0)
ret_hist = np.diff(np.interp(grid_h, t_hist, px_hist)) / tick_size

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
for ret, c, lbl in [(ret_hist, '#2196F3', 'Historical'), (ret_sim, '#F44336', 'MDQR')]:
    if len(ret) > 5:
        kde = stats.gaussian_kde(ret)
        x   = np.linspace(ret.min(), ret.max(), 300)
        ax.plot(x, kde(x), lw=2, color=c, label=lbl)
ax.set_xlabel('1-min return (ticks)'); ax.set_ylabel('Density')
ax.set_title('Return Distribution (KDE)'); ax.legend()

ax = axes[1]
for ret, c, lbl in [(ret_hist, '#2196F3', 'Historical'), (ret_sim, '#F44336', 'MDQR')]:
    if len(ret) > 10:
        (qt, qs_p), _ = stats.probplot(ret, dist='norm')
        ax.plot(qt, qs_p, '.', ms=3, color=c, label=lbl)
ax.plot([-4, 4], [-4, 4], 'k--', lw=1, label='Normal')
ax.set_xlabel('Theoretical $\mathcal{N}(0,1)$ quantiles')
ax.set_ylabel('Sample quantiles')
ax.set_title('Q–Q vs Normal'); ax.legend()

plt.suptitle('Figure 13 — 1-Minute Return Distribution', fontweight='bold')
plt.tight_layout(); plt.savefig('fig13_returns.pdf', dpi=150); plt.show()

print("=" * 60)
print("DIAGNOSTIC — Fig 13 (returns)")
for ret, lbl in [(ret_hist,'Historical'),(ret_sim,'MDQR')]:
    if len(ret) > 5:
        print(f"  {lbl}: n={len(ret):,}  mean={ret.mean():.4f}  std={ret.std():.4f}"
              f"  kurtosis={stats.kurtosis(ret):.2f}  skew={stats.skew(ret):.3f}")
    else:
        print(f"  {lbl}: insufficient returns ({len(ret)} samples)")
print("=" * 60)

---
## 13. Validation — Figures 14–15: Intraday Activity

We run $N_{\text{boot}} = 10$ independent simulations, each of length $T_{\text{sim}} = 6.5\,\text{h}$,
and aggregate event counts and traded volumes in 5-minute windows.  Box plots summarise the
distribution across runs, while the historical values are shown as individual dots.

A well-calibrated model should reproduce the characteristic **U-shaped intraday pattern**:
elevated activity at market open (09:30–10:30) and close (15:00–16:00) with a quieter midday
period.  This is captured by the hour embedding in $\mathbf{x}_k$.

In [ ]:
N_BOOT = 10
WINDOW = 300    # 5 min

def agg_5min(events, T_total=23400, window=300):
    n = int(T_total / window)
    counts = np.zeros(n); vols = np.zeros(n)
    for t, ti, si, lv, sz, _ in events:
        b = min(int(t / window), n - 1)
        counts[b] += 1; vols[b] += sz
    return counts, vols

hist_rel = [(float(r['time'] - df_norm['time'].min()), 0, 0, 1, float(r['size']), 0)
            for _, r in df_norm.iterrows()]
T_hist_range = float(df_norm['time'].max() - df_norm['time'].min())
hist_c, hist_v = agg_5min(hist_rel, T_total=T_hist_range)

print(f"Running {N_BOOT} bootstrap simulations …")
boot_c, boot_v = [], []
for b in range(N_BOOT):
    np.random.seed(200 + b); torch.manual_seed(200 + b)
    evs, _, _ = simulate_mdqr(
        mdqr_net, size_net, init_q, K,
        T_sim=23400.0, sz_bins=sz_bins, max_events=1_000_000, device=device
    )
    c, v = agg_5min(evs); boot_c.append(c); boot_v.append(v)
    if (b + 1) % 5 == 0: print(f"  {b+1}/{N_BOOT} done")

boot_c = np.array(boot_c); boot_v = np.array(boot_v)
n_bins = boot_c.shape[1]
from matplotlib.patches import Patch

def make_boxplot(boot_data, hist_data, ylabel, title, fname):
    fig, ax = plt.subplots(figsize=(16, 4))
    pos = np.arange(n_bins)
    ax.boxplot(boot_data.T, positions=pos + 0.15, widths=0.25, patch_artist=True,
               medianprops=dict(color='black', lw=2),
               boxprops=dict(facecolor='#F44336', alpha=0.7), showfliers=False)
    ax.scatter(pos - 0.15, hist_data[:n_bins], color='#2196F3', s=25, zorder=5)
    ticks = pos[::4]; ax.set_xticks(ticks)
    ax.set_xticklabels([f"{i*5}" for i in ticks], rotation=45, fontsize=8)
    ax.set_xlabel('Time from market open (min)'); ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend(handles=[
        Patch(facecolor='#F44336', alpha=0.7, label=f'MDQR (N={N_BOOT})'),
        plt.Line2D([0],[0], marker='o', color='#2196F3', lw=0, label='Historical')
    ], fontsize=9)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

make_boxplot(boot_c, hist_c, 'Event count', 'Figure 14 — Events per 5-min Window',
             'fig14_event_counts.pdf')
make_boxplot(boot_v, hist_v, 'Volume (norm.)', 'Figure 15 — Volume per 5-min Window',
             'fig15_volumes.pdf')

print("=" * 60)
print("DIAGNOSTIC — Figs 14-15 (box plots)")
print(f"  Hist events/5min: mean={hist_c.mean():.1f}  std={hist_c.std():.1f}")
print(f"  MDQR events/5min: mean={boot_c.mean():.1f}  std={boot_c.mean(axis=1).std():.1f}")
print(f"  Hist vol/5min:    mean={hist_v.mean():.1f}  std={hist_v.std():.1f}")
print(f"  MDQR vol/5min:    mean={boot_v.mean():.1f}  std={boot_v.mean(axis=1).std():.1f}")
# U-shape check: first bin vs middle bin
n_mid = n_bins // 2
print(f"  U-shape (hist):  first={hist_c[:2].mean():.1f}  mid={hist_c[n_mid-1:n_mid+1].mean():.1f}  last={hist_c[-2:].mean():.1f}")
print(f"  U-shape (MDQR):  first={boot_c[:,:2].mean():.1f}  mid={boot_c[:,n_mid-1:n_mid+1].mean():.1f}  last={boot_c[:,-2:].mean():.1f}")
print("=" * 60)

---
## 14. Validation — Figures 16–18: Order Size Distributions

The SizeNet models the conditional distribution $p(s \mid \eta, \ell, \mathbf{x})$ for
three conditioning dimensions:

- **Figure 16** — conditional on event *type* ($L$, $C$, $M$).  Market orders should
  be right-skewed (large trades), while cancellations may have smaller typical sizes.
- **Figure 17** — conditional on *level* (1, 3, 5) for limit orders.  Deeper levels
  often attract larger institutional orders, though with limited data this may not
  be clear-cut.
- **Figure 18** — marginal (stationary) distribution, obtained by averaging predictions
  over the validation set and comparing against the empirical distribution.

In [ ]:
def get_sizes_by(df, cond_col, cond_val):
    s = df[df[cond_col] == cond_val]['size'].dropna().astype(float).values
    return s[s > 0]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (et, name) in zip(axes, [('L','Limit'),('C','Cancel'),('M','Market')]):
    r = get_sizes_by(df_norm, 'type', et)
    s = get_sizes_by(df_sim,  'type', et)
    mx = np.percentile(r, 95) if len(r) > 0 else 10.0
    bins = np.linspace(0.5, mx, 40)
    if len(r) > 0: ax.hist(r, bins=bins, density=True, alpha=0.6, color='#2196F3', label='Historical')
    if len(s) > 0: ax.hist(s, bins=bins, density=True, alpha=0.6, color='#F44336', label='MDQR')
    ax.set_xlabel('Normalised size'); ax.set_title(name); ax.legend(fontsize=9)
plt.suptitle('Figure 16 — Size Distribution by Event Type', fontweight='bold')
plt.tight_layout(); plt.savefig('fig16_size_type.pdf', dpi=150); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, lv in zip(axes, [1, 3, 5]):
    r = df_norm[(df_norm['type']=='L') & (df_norm['lvl'].abs()==lv)]['size'].dropna().astype(float).values
    s = df_sim[ (df_sim['type']=='L')  & (df_sim['lvl'].abs()==lv)]['size'].dropna().astype(float).values
    r = r[r > 0]; s = s[s > 0]
    mx = np.percentile(r, 95) if len(r) > 0 else 10.0
    bins = np.linspace(0.5, mx, 40)
    if len(r) > 0: ax.hist(r, bins=bins, density=True, alpha=0.6, color='#2196F3', label='Historical')
    if len(s) > 0: ax.hist(s, bins=bins, density=True, alpha=0.6, color='#F44336', label='MDQR')
    ax.set_xlabel('Normalised size'); ax.set_title(f'Limit — Level {lv}'); ax.legend(fontsize=9)
plt.suptitle('Figure 17 — Limit Order Size by Level', fontweight='bold')
plt.tight_layout(); plt.savefig('fig17_size_level.pdf', dpi=150); plt.show()

# Figure 18: marginal distribution
r_all = df_norm['size'].dropna().astype(float).values; r_all = r_all[r_all > 0]
s_all = df_sim['size'].dropna().astype(float).values;  s_all = s_all[s_all > 0]
size_net.eval()
with torch.no_grad():
    n_samp = min(2000, len(Xc_s_va))
    avg_p  = size_net(Xc_s_va[:n_samp].to(device), Xh_s_va[:n_samp].to(device),
                      Xl_s_va[:n_samp].to(device), Xj_s_va[:n_samp].to(device)
                     ).cpu().numpy().mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
mx = np.percentile(r_all, 95) if len(r_all) > 0 else 10.0
bins = np.linspace(0.5, mx, 50)
if len(r_all) > 0: axes[0].hist(r_all, bins=bins, density=True, alpha=0.6, color='#2196F3', label='Historical')
if len(s_all) > 0: axes[0].hist(s_all, bins=bins, density=True, alpha=0.6, color='#F44336', label='MDQR sim.')
axes[0].bar(bin_mids, avg_p / (np.diff(sz_bins) + 1e-10), width=np.diff(sz_bins),
            alpha=0.35, color='purple', label='SizeNet (avg)')
axes[0].set_xlabel('Normalised size'); axes[0].set_ylabel('Density')
axes[0].set_title('Marginal Size Distribution'); axes[0].legend(fontsize=9)

if len(s_all) > 10 and len(r_all) > 10:
    qs2 = np.linspace(0.01, 0.99, 100)
    axes[1].plot(np.quantile(r_all, qs2), np.quantile(s_all, qs2), '.', color='#F44336', ms=4)
    d2 = [0, max(np.quantile(r_all, 0.99), np.quantile(s_all, 0.99))]
    axes[1].plot(d2, d2, 'k--', lw=1)
axes[1].set_xlabel('Historical quantiles'); axes[1].set_ylabel('MDQR quantiles')
axes[1].set_title('Q–Q: Marginal Size')
plt.suptitle('Figure 18 — Stationary Order Size Distribution', fontweight='bold')
plt.tight_layout(); plt.savefig('fig18_size_marginal.pdf', dpi=150); plt.show()

print("=" * 60)
print("DIAGNOSTIC — Figs 16-18 (size distributions)")
for t in ['L','C','M']:
    r = get_sizes_by(df_norm, 'type', t); s = get_sizes_by(df_sim, 'type', t)
    if len(r) > 0 and len(s) > 0:
        print(f"  {t}: real mean={r.mean():.3f} med={np.median(r):.3f} | "
              f"sim  mean={s.mean():.3f} med={np.median(s):.3f}")
print("=" * 60)

---
## 15. Summary

### What the MDQR model achieves over DQR

The MDQR model relaxes three independence assumptions that are hard-coded into the per-queue DQR:

1. **Cross-level intensities.** Because all $2K$ queue sizes enter $\mathbf{x}_k$, the arrival
   rate at level $i$ can depend on the state of levels $j \neq i$.  This is structurally
   impossible for single-queue DQR models.

2. **Joint calibration of $3 \times 2K = 30$ intensities.** One shared network captures how
   the full LOB state drives event activity, enabling the model to learn, for example, that
   a thin ask queue at level 1 simultaneously raises the probability of a buy market order and
   lowers the probability of new ask limit orders.

3. **Order size distribution.** SizeNet provides a complete conditional distribution over
   normalised order sizes, replacing the unit-size assumption of QR/DQR.

### Checklist against the paper (Section 4)

| Specification | This notebook |
|---------------|---------------|
| Architecture [256, 64], Tanh, ReLU/Softmax (Table 4) | ✓ |
| Global $\Delta t_k$: training clip $[1\,\mu\text{s}, 30\,\text{s}]$; timing metric filter $> 1\,\text{ms}$ | ✓ |
| SizeNet with categorical cross-entropy (Section 4.1) | ✓ |
| Adam + CAWR $[10^{-5}, 10^{-3}]$, patience 30 (single-day adaptation from paper's 10) | ✓ |
| Random 80/20 train/val split | ✓ |
| Figures 5, 9–18 | ✓ |

### Known limitations & Deviations from the Paper

Although this implementation of the MDQR model successfully incorporates the core innovations of Bodor & Carlier (2025) — including joint queue modeling, Trade Imbalances ($TI_\tau$), and exact conditional order-size distributions — a few structural adaptations were necessary:

1. **Dataset Size and Asset Class:** The original paper calibrates the model on 3 months of Eurex Bund futures data (highly liquid, thick queues). This notebook uses a single trading day of NASDAQ INTC equities (thinner queues, different microstructural dynamics). Consequently, the model may exhibit higher variance and requires stronger regularization/patience.

2. **Global vs. Local Event Embedding ($e(t_k)$):**
   The paper tracks the most recent event type *at each specific price level* $e_i(t_k)$ to feed into the network. To prevent severe overfitting on a single-day dataset, this implementation simplifies the state space by tracking only the single, global most recent event $e(t_k)$ across the entire limit order book.

3. **Early Stopping Patience:**
   The paper utilizes a patience of 10 epochs for early stopping. Due to the high noise associated with single-day equity data, we increased the patience to 30 epochs (`PATIENCE = 30`) to prevent premature termination during training.

4. **Simulation Engine & Price Dynamics:**
   Our Gillespie simulation engine uses a simplified rule for mid-price updates: the price shifts only when a level-1 queue is completely depleted. A fully realistic matching engine (as hinted in the paper) would also process limit orders placed *inside* the bid-ask spread, directly narrowing the spread and updating the mid-price without requiring full depletion.